In [21]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =====================================================
# PATHS
# =====================================================

BOOK_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv"
MOVIE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Movie /data/df_final.csv"
MOBILE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0316_Single/data/data/mobilerec_final.csv"

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/state of the art/paper_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# ================== BOOK DATA =========================
# =====================================================

df_books = pd.read_csv(BOOK_PATH)

genres_split = df_books["genres"].astype(str).str.split(",", n=1, expand=True)
df_books["g1"] = genres_split[0].str.strip()
df_books["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

books = df_books[["book_id", "g1", "g2"]].drop_duplicates()

first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

summary_books = pd.DataFrame({
    "genre": all_genres,
    "total": (first_counts.reindex(all_genres, fill_value=0) +
              second_counts.reindex(all_genres, fill_value=0)).values
}).sort_values("total", ascending=False)

# =====================================================
# ================== MOVIE DATA ========================
# =====================================================

df_movies = pd.read_csv(MOVIE_PATH)

df_movies["decade"] = df_movies["decade"].fillna(-1).astype(int).astype(str)
df_movies = df_movies[df_movies["decade"] != "-1"]

movies = df_movies[["item_id", "decade"]].drop_duplicates()

summary_movies = movies["decade"].value_counts().reset_index()
summary_movies.columns = ["decade", "total"]
summary_movies = summary_movies.sort_values("total", ascending=False)

# =====================================================
# ================== MOBILE DATA =======================
# =====================================================

df_mobile = pd.read_csv(MOBILE_PATH, low_memory=False)

category_mapping = {
"Action":"Games","Puzzle":"Games","Simulation":"Games","Strategy":"Games","Casual":"Games",
"Adventure":"Games","Sports":"Games","Arcade":"Games","Casino":"Games","Card":"Games",
"Board":"Games","Word":"Games","Racing":"Games","Role Playing":"Games","Trivia":"Games",
"Education":"Education & Learning","Educational":"Education & Learning",
"Books & Reference":"Education & Learning","Libraries & Demo":"Education & Learning",
"Health & Fitness":"Health & Lifestyle","Lifestyle":"Health & Lifestyle",
"Beauty":"Health & Lifestyle","Parenting":"Health & Lifestyle",
"Entertainment":"Media & Entertainment","Music":"Media & Entertainment",
"Music & Audio":"Media & Entertainment","Video Players & Editors":"Media & Entertainment",
"Comics":"Media & Entertainment",
"Social":"Social & Communication","Communication":"Social & Communication",
"Dating":"Social & Communication",
"Shopping":"Shopping & Food","Food & Drink":"Shopping & Food",
"Travel & Local":"Travel & Navigation","Maps & Navigation":"Travel & Navigation",
"Auto & Vehicles":"Travel & Navigation",
"Productivity":"Work & Productivity","Business":"Work & Productivity",
"Finance":"Work & Productivity","Tools":"Work & Productivity",
"Weather":"Utilities & Info","News & Magazines":"Utilities & Info",
"Photography":"Utilities & Info","Art & Design":"Utilities & Info",
"Personalization":"Utilities & Info","House & Home":"Utilities & Info",
"Events":"Utilities & Info"
}

df_mobile["category_mapped"] = df_mobile["app_category"].map(category_mapping)

mapped_counts = (
    df_mobile[["app_package","category_mapped"]]
    .drop_duplicates()["category_mapped"]
    .value_counts()
    .sort_values(ascending=False)
)

# =====================================================
# ================== COMMON STYLE ======================
# =====================================================

def apply_style(ax):
    ax.yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("black")

# =====================================================
# ================== FIGURE 1: BOOKS ===================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(summary_books)) * 1.1

ax.bar(x, summary_books["total"],
       width=0.45, color="#A8D5BA", edgecolor="#6FAF8F")

ax.set_title("Books by Genre", fontsize=12)          # +1
ax.set_xlabel("Genre", fontsize=11.5)                 # +1.5
ax.set_ylabel("Number of Unique Books", fontsize=11.5)

ax.set_xticks(x)
ax.set_xticklabels(summary_books["genre"], rotation=60, fontsize=10)  # +1

ax.tick_params(axis='y', labelsize=10)  # +1

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "books_genre.png", dpi=600)
plt.close()

# =====================================================
# ================== FIGURE 2: MOVIES ==================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(summary_movies)) * 1.1

ax.bar(x, summary_movies["total"],
       width=0.4, color="#CDB4DB", edgecolor="#7B5FA5")

ax.set_title("Movies by Decade", fontsize=12)
ax.set_xlabel("Decade", fontsize=11.5)
ax.set_ylabel("Number of Unique Movies", fontsize=11.5)

ax.set_xticks(x)
ax.set_xticklabels(summary_movies["decade"], rotation=60, fontsize=9)  # +1

ax.tick_params(axis='y', labelsize=10)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "movies_decade.png", dpi=600)
plt.close()

# =====================================================
# ================== FIGURE 3: MOBILE ==================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(mapped_counts)) * 1.1

ax.bar(x, mapped_counts.values,
       width=0.4, color="#FFD166", edgecolor="#E6C200")

ax.set_title("Mobile Apps by Category", fontsize=12)
ax.set_xlabel("category", fontsize=11.5)
ax.set_ylabel("Number of Unique Applications", fontsize=11.5)

ax.set_xticks(x)
ax.set_xticklabels(mapped_counts.index, rotation=60, fontsize=9)

ax.tick_params(axis='y', labelsize=10)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "mobile_domains.png", dpi=600)
plt.close()

print("Saved all three figures to:")
print(OUT_DIR)

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =====================================================
# PATHS (UPDATED)
# =====================================================

BOOK_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv"
MOVIE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Movie /data/df_final.csv"
MOBILE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0316_Single/data/data/mobilerec_final.csv"

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/state of the art/paper_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# ================== BOOK DATA =========================
# =====================================================

df_books = pd.read_csv(BOOK_PATH)

genres_split = df_books["genres"].astype(str).str.split(",", n=1, expand=True)
df_books["g1"] = genres_split[0].str.strip()
df_books["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

books = df_books[["book_id", "g1", "g2"]].drop_duplicates()

first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

summary_books = pd.DataFrame({
    "genre": all_genres,
    "total": (first_counts.reindex(all_genres, fill_value=0) +
              second_counts.reindex(all_genres, fill_value=0)).values
}).sort_values("total", ascending=False)

# =====================================================
# ================== MOVIE DATA ========================
# =====================================================

df_movies = pd.read_csv(MOVIE_PATH)

df_movies["decade"] = df_movies["decade"].fillna(-1).astype(int).astype(str)
df_movies = df_movies[df_movies["decade"] != "-1"]

movies = df_movies[["item_id", "decade"]].drop_duplicates()

summary_movies = movies["decade"].value_counts().reset_index()
summary_movies.columns = ["decade", "total"]
summary_movies = summary_movies.sort_values("total", ascending=False)

# =====================================================
# ================== MOBILE DATA =======================
# =====================================================

df_mobile = pd.read_csv(MOBILE_PATH, low_memory=False)

category_mapping = {
"Action":"Games","Puzzle":"Games","Simulation":"Games","Strategy":"Games","Casual":"Games",
"Adventure":"Games","Sports":"Games","Arcade":"Games","Casino":"Games","Card":"Games",
"Board":"Games","Word":"Games","Racing":"Games","Role Playing":"Games","Trivia":"Games",
"Education":"Education & Learning","Educational":"Education & Learning",
"Books & Reference":"Education & Learning","Libraries & Demo":"Education & Learning",
"Health & Fitness":"Health & Lifestyle","Lifestyle":"Health & Lifestyle",
"Beauty":"Health & Lifestyle","Parenting":"Health & Lifestyle",
"Entertainment":"Media & Entertainment","Music":"Media & Entertainment",
"Music & Audio":"Media & Entertainment","Video Players & Editors":"Media & Entertainment",
"Comics":"Media & Entertainment",
"Social":"Social & Communication","Communication":"Social & Communication",
"Dating":"Social & Communication",
"Shopping":"Shopping & Food","Food & Drink":"Shopping & Food",
"Travel & Local":"Travel & Navigation","Maps & Navigation":"Travel & Navigation",
"Auto & Vehicles":"Travel & Navigation",
"Productivity":"Work & Productivity","Business":"Work & Productivity",
"Finance":"Work & Productivity","Tools":"Work & Productivity",
"Weather":"Utilities & Info","News & Magazines":"Utilities & Info",
"Photography":"Utilities & Info","Art & Design":"Utilities & Info",
"Personalization":"Utilities & Info","House & Home":"Utilities & Info",
"Events":"Utilities & Info"
}

df_mobile["category_mapped"] = df_mobile["app_category"].map(category_mapping)

mapped_counts = (
    df_mobile[["app_package","category_mapped"]]
    .drop_duplicates()["category_mapped"]
    .value_counts()
    .sort_values(ascending=False)
)

# =====================================================
# ================== PLOTTING ==========================
# =====================================================

# A4 width ≈ 8.27 inches → use ~7.5 for margins
fig, axes = plt.subplots(1, 3, figsize=(12,5))

# ---------------- BOOKS ----------------
x_books = np.arange(len(summary_books)) * 1.1

axes[0].bar(x_books, summary_books["total"],
            width=0.45, color="#A8D5BA", edgecolor="#6FAF8F")

axes[0].set_title("Books by Genre", fontsize=10)
axes[0].set_xlabel("Genre", fontsize=9)
axes[0].set_ylabel("Number of Unique Books", fontsize=9)

axes[0].set_xticks(x_books)
axes[0].set_xticklabels(summary_books["genre"], rotation=60, fontsize=7)

axes[0].tick_params(axis='y', labelsize=7)
axes[0].yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

# ADD FULL BORDER
for spine in axes[0].spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
    spine.set_color("black")

# ---------------- MOVIES ----------------
x_movies = np.arange(len(summary_movies)) * 1.1

axes[1].bar(x_movies, summary_movies["total"],
            width=0.4, color="#CDB4DB", edgecolor="#7B5FA5")

axes[1].set_title("Movies by Decade", fontsize=10)
axes[1].set_xlabel("Decade", fontsize=9)
axes[1].set_ylabel("Number of Unique Movies", fontsize=9)

axes[1].set_xticks(x_movies)
axes[1].set_xticklabels(summary_movies["decade"], rotation=60, fontsize=7)

axes[1].tick_params(axis='y', labelsize=7)
axes[1].yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

for spine in axes[1].spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
    spine.set_color("black")

# ---------------- MOBILE ----------------
x_mobile = np.arange(len(mapped_counts)) * 1.1

axes[2].bar(x_mobile, mapped_counts.values,
            width=0.4, color="#FFD166", edgecolor="#E6C200")

axes[2].set_title("Mobile Apps by Domain", fontsize=10)
axes[2].set_xlabel("Domain", fontsize=9)
axes[2].set_ylabel("Number of Unique Applications", fontsize=9)

axes[2].set_xticks(x_mobile)
axes[2].set_xticklabels(mapped_counts.index, rotation=60, fontsize=7)

axes[2].tick_params(axis='y', labelsize=7)
axes[2].yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

for spine in axes[2].spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
    spine.set_color("black")

# =====================================================
# LAYOUT (MORE SPACE BETWEEN PANELS)
# =====================================================

plt.tight_layout(pad=1.2, w_pad=2.5)

# =====================================================
# SAVE
# =====================================================

plt.savefig(OUT_DIR / "combined_3plots_A4.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved combined figure to:")
print(OUT_DIR)

Saved combined figure to:
/home/moshtasa/Research/phd-svd-recsys/state of the art/paper_figures


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =====================================================
# PATHS
# =====================================================

BOOK_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv"
MOVIE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Movie /data/df_final.csv"
MOBILE_PATH = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0316_Single/data/data/mobilerec_final.csv"

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/state of the art/paper_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# ================== BOOK DATA =========================
# =====================================================

df_books = pd.read_csv(BOOK_PATH)

genres_split = df_books["genres"].astype(str).str.split(",", n=1, expand=True)
df_books["g1"] = genres_split[0].str.strip()
df_books["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

books = df_books[["book_id", "g1", "g2"]].drop_duplicates()

first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

summary_books = pd.DataFrame({
    "genre": all_genres,
    "total": (first_counts.reindex(all_genres, fill_value=0) +
              second_counts.reindex(all_genres, fill_value=0)).values
}).sort_values("total", ascending=False)

# =====================================================
# ================== MOVIE DATA ========================
# =====================================================

df_movies = pd.read_csv(MOVIE_PATH)

df_movies["decade"] = df_movies["decade"].fillna(-1).astype(int).astype(str)
df_movies = df_movies[df_movies["decade"] != "-1"]

movies = df_movies[["item_id", "decade"]].drop_duplicates()

summary_movies = movies["decade"].value_counts().reset_index()
summary_movies.columns = ["decade", "total"]
summary_movies = summary_movies.sort_values("total", ascending=False)

# =====================================================
# ================== MOBILE DATA =======================
# =====================================================

df_mobile = pd.read_csv(MOBILE_PATH, low_memory=False)

category_mapping = {
"Action":"Games","Puzzle":"Games","Simulation":"Games","Strategy":"Games","Casual":"Games",
"Adventure":"Games","Sports":"Games","Arcade":"Games","Casino":"Games","Card":"Games",
"Board":"Games","Word":"Games","Racing":"Games","Role Playing":"Games","Trivia":"Games",
"Education":"Education & Learning","Educational":"Education & Learning",
"Books & Reference":"Education & Learning","Libraries & Demo":"Education & Learning",
"Health & Fitness":"Health & Lifestyle","Lifestyle":"Health & Lifestyle",
"Beauty":"Health & Lifestyle","Parenting":"Health & Lifestyle",
"Entertainment":"Media & Entertainment","Music":"Media & Entertainment",
"Music & Audio":"Media & Entertainment","Video Players & Editors":"Media & Entertainment",
"Comics":"Media & Entertainment",
"Social":"Social & Communication","Communication":"Social & Communication",
"Dating":"Social & Communication",
"Shopping":"Shopping & Food","Food & Drink":"Shopping & Food",
"Travel & Local":"Travel & Navigation","Maps & Navigation":"Travel & Navigation",
"Auto & Vehicles":"Travel & Navigation",
"Productivity":"Work & Productivity","Business":"Work & Productivity",
"Finance":"Work & Productivity","Tools":"Work & Productivity",
"Weather":"Utilities & Info","News & Magazines":"Utilities & Info",
"Photography":"Utilities & Info","Art & Design":"Utilities & Info",
"Personalization":"Utilities & Info","House & Home":"Utilities & Info",
"Events":"Utilities & Info"
}

df_mobile["category_mapped"] = df_mobile["app_category"].map(category_mapping)

mapped_counts = (
    df_mobile[["app_package","category_mapped"]]
    .drop_duplicates()["category_mapped"]
    .value_counts()
    .sort_values(ascending=False)
)

# =====================================================
# ================== COMMON STYLE ======================
# =====================================================

def apply_style(ax):
    ax.yaxis.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("black")

# =====================================================
# ================== FIGURE 1: BOOKS ===================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(summary_books)) * 1.1

ax.bar(x, summary_books["total"],
       width=0.45, color="#A8D5BA", edgecolor="#6FAF8F")

ax.set_title("Books by Genre", fontsize=11)          # +1
ax.set_xlabel("Genre", fontsize=10.5)                 # +1.5
ax.set_ylabel("Number of Unique Books", fontsize=10.5)

ax.set_xticks(x)
ax.set_xticklabels(summary_books["genre"], rotation=60, fontsize=9)  # +1

ax.tick_params(axis='y', labelsize=9)  # +1

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "books_genre.png", dpi=600)
plt.close()

# =====================================================
# ================== FIGURE 2: MOVIES ==================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(summary_movies)) * 1.1

ax.bar(x, summary_movies["total"],
       width=0.4, color="#CDB4DB", edgecolor="#7B5FA5")

ax.set_title("Movies by Decade", fontsize=11)
ax.set_xlabel("Decade", fontsize=10.5)
ax.set_ylabel("Number of Unique Movies", fontsize=10.5)

ax.set_xticks(x)
ax.set_xticklabels(summary_movies["decade"], rotation=60, fontsize=8)  # +1

ax.tick_params(axis='y', labelsize=9)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "movies_decade.png", dpi=600)
plt.close()

# =====================================================
# ================== FIGURE 3: MOBILE ==================
# =====================================================

fig, ax = plt.subplots(figsize=(5, 3))

x = np.arange(len(mapped_counts)) * 1.1

ax.bar(x, mapped_counts.values,
       width=0.4, color="#FFD166", edgecolor="#E6C200")

ax.set_title("Mobile Apps by Category", fontsize=11)
ax.set_xlabel("Category", fontsize=10.5)
ax.set_ylabel("Number of Unique Applications", fontsize=10.5)

ax.set_xticks(x)
ax.set_xticklabels(mapped_counts.index, rotation=60, fontsize=8)

ax.tick_params(axis='y', labelsize=9)

apply_style(ax)

plt.tight_layout()
plt.savefig(OUT_DIR / "mobile_domains.png", dpi=600)
plt.close()

print("Saved all three figures to:")
print(OUT_DIR)